# Great Expectations Setup

Defines validation suites for all three raw data files.
These suites run as pipeline gates before any transformation begins.
If a file fails validation, processing stops with a clear error.

Suites defined:
- general_ema_raw: EMA survey file
- sensing_raw: mobile sensing file
- covid_ema_raw: COVID EMA survey file

In [5]:
import great_expectations as gx
import pandas as pd
import numpy as np
from pathlib import Path

print(f"GE version: {gx.__version__}")

DATA_DIR = Path("../../data/raw/college_experience_dataset")

context = gx.get_context(mode="ephemeral")
print(type(context))

GE version: 1.16.0
<class 'great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext'>


---
### Suite 1: general_ema_raw

Validates the raw EMA survey file.

In [8]:
df_ema = pd.read_csv(DATA_DIR / "EMA" / "general_ema.csv")
print(f"Shape: {df_ema.shape}")


# Wire up GE 1.x datasource
datasource_ema = context.data_sources.add_or_update_pandas("general_ema_source")
asset_ema = datasource_ema.add_dataframe_asset("general_ema_asset")
batch_def_ema = asset_ema.add_batch_definition_whole_dataframe("general_ema_batch")
batch_ema = batch_def_ema.get_batch(batch_parameters={"dataframe": df_ema})

# Build expectation suite
suite_ema = gx.ExpectationSuite(name="general_ema_raw")

# Identity columns
suite_ema.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="uid")
)
suite_ema.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="day")
)
suite_ema.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="day", min_value=20170101, max_value=20221231
    )
)

# PHQ4 items
for col in ["phq4-1", "phq4-2", "phq4-3", "phq4-4"]:
    suite_ema.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=col, min_value=0, max_value=3, mostly=0.99
        )
    )

# PHQ4 score
suite_ema.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="phq4_score", min_value=0, max_value=12, mostly=0.99
    )
)

# SSE items
for col in ["sse3-1", "sse3-2", "sse3-3", "sse3-4"]:
    suite_ema.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=col, min_value=1, max_value=5, mostly=0.99
        )
    )

# Single-item scales
suite_ema.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="stress", min_value=1, max_value=5, mostly=0.99
    )
)
suite_ema.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="social_level", min_value=1, max_value=5, mostly=0.99
    )
)

# Table-level
suite_ema.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=200_000, max_value=250_000
    )
)
suite_ema.add_expectation(
    gx.expectations.ExpectTableColumnCountToBeBetween(
        min_value=15, max_value=25
    )
)

# Validate
results_ema = batch_ema.validate(suite_ema)
passed = results_ema["statistics"]["successful_expectations"]
total  = results_ema["statistics"]["evaluated_expectations"]
status = "PASS" if results_ema["success"] else "FAIL"
print(f"\n[{status}] general_ema_raw: {passed}/{total} expectations passed")

if not results_ema["success"]:
    for r in results_ema["results"]:
        if not r["success"]:
            col = r["expectation_config"].get("kwargs", {}).get("column", "table-level")
            exp = r["expectation_config"]["type"]
            print(f"  FAILED: {exp} on '{col}'")

Shape: (217155, 19)


Calculating Metrics:   0%|          | 0/97 [00:00<?, ?it/s]


[PASS] general_ema_raw: 16/16 expectations passed


---
### Suite 2: sensing_raw

Validates the raw sensing file.

In [9]:
df_sens = pd.read_csv(DATA_DIR / "Sensing" / "sensing.csv")
df_sens = df_sens.copy()
print(f"Shape: {df_sens.shape}")

datasource_sens = context.data_sources.add_or_update_pandas("sensing_source")
asset_sens = datasource_sens.add_dataframe_asset("sensing_asset")
batch_def_sens = asset_sens.add_batch_definition_whole_dataframe("sensing_batch")
batch_sens = batch_def_sens.get_batch(batch_parameters={"dataframe": df_sens})

suite_sens = gx.ExpectationSuite(name="sensing_raw")

# Identity
suite_sens.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="uid")
)
suite_sens.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="day")
)
suite_sens.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="day", min_value=20170101, max_value=20221231
    )
)

# Platform flag
suite_sens.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="is_ios", value_set=[0, 1], mostly=0.99
    )
)

# Quality columns (hours of coverage, 0-24)
for col in ["quality_activity", "quality_loc"]:
    if col in df_sens.columns:
        suite_sens.add_expectation(
            gx.expectations.ExpectColumnValuesToBeBetween(
                column=col, min_value=0, max_value=24, mostly=0.99
            )
        )

# Sleep duration (hours, 0-24)
suite_sens.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="sleep_duration", min_value=0, max_value=24, mostly=0.99
    )
)

# Activity still (seconds, 0-86400)
# Values of exactly 86400 are sensor failures - valid range but
# filtered during feature engineering (D_S07)
suite_sens.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="act_still_ep_0", min_value=0, max_value=86400, mostly=0.99
    )
)

# Table-level
suite_sens.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=200_000, max_value=230_000
    )
)
suite_sens.add_expectation(
    gx.expectations.ExpectTableColumnCountToBeBetween(
        min_value=600, max_value=700
    )
)

results_sens = batch_sens.validate(suite_sens)
passed = results_sens["statistics"]["successful_expectations"]
total  = results_sens["statistics"]["evaluated_expectations"]
status = "PASS" if results_sens["success"] else "FAIL"
print(f"\n[{status}] sensing_raw: {passed}/{total} expectations passed")

if not results_sens["success"]:
    for r in results_sens["results"]:
        if not r["success"]:
            col = r["expectation_config"].get("kwargs", {}).get("column", "table-level")
            exp = r["expectation_config"]["type"]
            print(f"  FAILED: {exp} on '{col}'")

Shape: (216065, 651)


Calculating Metrics:   0%|          | 0/57 [00:00<?, ?it/s]


[PASS] sensing_raw: 10/10 expectations passed


---
### Suite 3: covid_ema_raw

Validates the raw COVID EMA file.

In [10]:
df_covid = pd.read_csv(DATA_DIR / "EMA" / "covid_ema.csv")
print(f"Shape: {df_covid.shape}")

datasource_covid = context.data_sources.add_or_update_pandas("covid_ema_source")
asset_covid = datasource_covid.add_dataframe_asset("covid_ema_asset")
batch_def_covid = asset_covid.add_batch_definition_whole_dataframe("covid_ema_batch")
batch_covid = batch_def_covid.get_batch(batch_parameters={"dataframe": df_covid})

suite_covid = gx.ExpectationSuite(name="covid_ema_raw")

# Identity
suite_covid.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="uid")
)
suite_covid.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="day")
)

# COVID data starts March 2020
suite_covid.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="day", min_value=20200101, max_value=20221231
    )
)

# All 10 COVID items use 1-7 scale
for i in range(1, 11):
    col = f"COVID-{i}"
    if col in df_covid.columns:
        suite_covid.add_expectation(
            gx.expectations.ExpectColumnValuesToBeBetween(
                column=col, min_value=1, max_value=7, mostly=0.99
            )
        )

# Table-level
suite_covid.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=14_000, max_value=20_000
    )
)

results_covid = batch_covid.validate(suite_covid)
passed = results_covid["statistics"]["successful_expectations"]
total  = results_covid["statistics"]["evaluated_expectations"]
status = "PASS" if results_covid["success"] else "FAIL"
print(f"\n[{status}] covid_ema_raw: {passed}/{total} expectations passed")

if not results_covid["success"]:
    for r in results_covid["results"]:
        if not r["success"]:
            col = r["expectation_config"].get("kwargs", {}).get("column", "table-level")
            exp = r["expectation_config"]["type"]
            print(f"  FAILED: {exp} on '{col}'")

Shape: (16511, 12)


Calculating Metrics:   0%|          | 0/89 [00:00<?, ?it/s]


[PASS] covid_ema_raw: 14/14 expectations passed


In [12]:
all_passed = (
    results_ema["success"] and
    results_sens["success"] and
    results_covid["success"]
)

ema_p   = results_ema["statistics"]["successful_expectations"]
ema_t   = results_ema["statistics"]["evaluated_expectations"]
sens_p  = results_sens["statistics"]["successful_expectations"]
sens_t  = results_sens["statistics"]["evaluated_expectations"]
covid_p = results_covid["statistics"]["successful_expectations"]
covid_t = results_covid["statistics"]["evaluated_expectations"]

print("=" * 50)
print("VALIDATION SUMMARY")
print("=" * 50)
print(f"  general_ema_raw : {'PASS' if results_ema['success']   else 'FAIL'} ({ema_p}/{ema_t})")
print(f"  sensing_raw     : {'PASS' if results_sens['success']  else 'FAIL'} ({sens_p}/{sens_t})")
print(f"  covid_ema_raw   : {'PASS' if results_covid['success'] else 'FAIL'} ({covid_p}/{covid_t})")
print()
if all_passed:
    print("All suites passed. Safe to proceed to label engineering.")
else:
    print("One or more suites FAILED. Investigate before proceeding.")

VALIDATION SUMMARY
  general_ema_raw : PASS (16/16)
  sensing_raw     : PASS (10/10)
  covid_ema_raw   : PASS (14/14)

All suites passed. Safe to proceed to label engineering.
